In [585]:
!pip install datasets==2.16.0 huggingface-hub==0.34.0 scikit-learn jenga pandas numpy setuptools nltk category_encoders ftfy cleanlab seaborn -q

## Create Directories

In [586]:
import os

current_dir = os.getcwd()
if 'notebook' in current_dir:
    BASE_DIR = os.path.dirname(current_dir)
else:
    BASE_DIR = current_dir

os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)

# Create folder structure
os.makedirs('results/', exist_ok=True)
os.makedirs('figures/', exist_ok=True)

RESULTS_DIR = os.path.join(BASE_DIR, "results/")
FIGURES_DIR = os.path.join(BASE_DIR, "figures/")

print(f"Working directory: {BASE_DIR}")
print(f"Folder structure created")

Working directory: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1
Folder structure created


## Imports

### Import libraries

In [587]:
# Import libraries

import os, sys
import pandas as pd
import numpy as np
from datetime import datetime
import gc
import time
from contextlib import contextmanager

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error
from scipy.stats import ttest_rel, ks_2samp, norm
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import ast
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
import tracemalloc


import sklearn
sklearn.set_config(enable_metadata_routing=True)

import warnings
warnings.filterwarnings("ignore")

### Import corruption & cleaning modules


In [588]:
sys.path.append(os.path.join(BASE_DIR, "python_scripts/"))

# Import unified corruption functions
import corruption as corrupt

# Import unified cleaning functions
import cleaning as clean

print("Imported corruption & cleaning modules")

Imported corruption & cleaning modules


## Building the Model

### Configurations

In [589]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
SAMPLES_PER_CLASS = 20000
EXPECTED_TOTAL = SAMPLES_PER_CLASS * 5
TARGET_COL = 'income'

### Preprocessing and Model Functions

In [590]:
NUMERIC_FEATURES = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
CATEGORICAL_FEATURES = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender']

def build_model():
    """Build LogisticRegression with StandardScaler + OneHotEncoder"""
    numeric_transformer =  StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, NUMERIC_FEATURES),
            ('cat', categorical_transformer, CATEGORICAL_FEATURES)
        ]
    )

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
    ])

    return model

## Utility Functions

In [591]:
def clear_memory():
    _ = gc.collect()

@contextmanager
def timer():
    """Context manager to time a block of code"""
    start = time.perf_counter()
    yield lambda: time.perf_counter() - start

def prepare_baseline_data(X, y):
    # Ensure numeric columns are numeric
    X = X.copy()
    X[NUMERIC_FEATURES] = X[NUMERIC_FEATURES].apply(pd.to_numeric, errors='coerce')

    # Drop rows with NaN in numeric features or target
    mask = X[NUMERIC_FEATURES].notna().all(axis=1) & y.notna()
    X = X[mask].reset_index(drop=True)
    y = y[mask].reset_index(drop=True)

    # Add row_id for fixed splits
    X["row_id"] = X.index

    return X, y 

def resample_balanced(df, target_col, n_per_class=10000):
    """Balance dataset to n_per_class samples per class"""
    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].copy()

    # Separate classes
    class_0 = df[df[target_col] == df[target_col].unique()[0]]
    class_1 = df[df[target_col] == df[target_col].unique()[1]]

    # Resample each class
    class_0_resampled = resample(
        class_0,
        replace=False,
        n_samples=min(n_per_class, len(class_0)),
        random_state=RANDOM_STATE
    )
    class_1_resampled = resample(
        class_1,
        replace=False,
        n_samples=min(n_per_class, len(class_1)),
        random_state=RANDOM_STATE
    )

    # Combine and shuffle
    df_balanced = pd.concat([class_0_resampled, class_1_resampled]).sample(frac=1, random_state=RANDOM_STATE)

    X = df_balanced.drop(columns=[target_col])
    y = df_balanced[target_col]

    return X, y


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    }

## Loading the Data

In [592]:
print("Loading the raw dataset...")
df_raw = pd.read_csv("adult_income_cleaned.csv")
print(f"Loaded: {len(df_raw):,} rows")

Loading the raw dataset...
Loaded: 48,842 rows


In [593]:
df_raw = df_raw.groupby(TARGET_COL, group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), SAMPLES_PER_CLASS), random_state=9)
)

print(f"Sampled to Balanced Pool: {len(df_raw):,} total rows")
print(f"Class Distribution: {df_raw[TARGET_COL].value_counts().to_dict()}")

Sampled to Balanced Pool: 31,687 total rows
Class Distribution: {'<=50K': 20000, '>50K': 11687}


## Training and Evaluation Function

In [594]:
def train_and_evaluate(X_train, X_test, y_train, y_test, train_ids, test_ids):
    # Use fixed IDs for membership
    X_train_split = X_train[X_train["row_id"].isin(train_ids)].copy()
    X_test_split  = X_test[X_test["row_id"].isin(test_ids)].copy()
    y_train_split = y_train[X_train["row_id"].isin(train_ids)].copy()
    y_test_split  = y_test[X_test["row_id"].isin(test_ids)].copy()

    # Drop row_id for training
    X_train_clean = X_train_split.drop(columns=["row_id"])
    X_test_clean  = X_test_split.drop(columns=["row_id"])

    # Convert to numeric (coerce corrupted strings like '50.0x' to NaN)
    X_train_clean = X_train_clean.apply(pd.to_numeric, errors='coerce')
    X_test_clean = X_test_clean.apply(pd.to_numeric, errors='coerce')

    # Fill NaN with 0 (simple strategy for eval)
    X_train_clean = X_train_clean.fillna(0)
    X_test_clean  = X_test_clean.fillna(0)

    if len(X_train_clean) < 20 or len(X_test_clean) < 10:
        return None, {"error": "insufficient_data"}

    # Train & evaluate
    model = build_model()
    model.fit(X_train_clean, y_train_split)
    y_pred = model.predict(X_test_clean)

    metrics = compute_metrics(y_test_split, y_pred)
    stats = {
        "train_total": len(X_train_split),
        "train_valid": len(X_train_clean),
        "test_total": len(X_test_split),
        "test_valid": len(X_test_clean),
    }

    del model, X_train_clean, X_test_clean
    clear_memory()

    return metrics, stats

## Running the Model on Baseline Data

### Prepare baseline data

In [595]:
X, y = resample_balanced(df_raw, TARGET_COL, n_per_class=10000)
#combine x and y to prepare baseline data
df_baseline = pd.concat([X, y], axis=1)
X_baseline, y_baseline = prepare_baseline_data(X, y)

### Run the model on baseline data

In [596]:
# create fixed train/test split
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_baseline, y_baseline, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_baseline
)
train_ids = set(X_train_clean["row_id"].tolist())
test_ids = set(X_test_clean["row_id"].tolist())
print(f"Split: {len(X_train_clean):,} train / {len(X_test_clean):,} test")

# train and evaluate on baseline data
print("Training the model on baseline data...")
with timer() as t:
    metrics_base, stats_base = train_and_evaluate(X_train_clean, X_test_clean, y_train_clean, y_test_clean, train_ids, test_ids)
baseline_acc = metrics_base['accuracy']
print(f"   Baseline Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"   F1: {metrics_base['f1']:.4f}")

# save baseline results
baseline_info = {
    "timestamp": datetime.now().isoformat(),
    "dataset": "Adult Income",
    "dataset_size": len(X_baseline),
    "train_size": len(train_ids),
    "test_size": len(test_ids),
    "baseline_accuracy": baseline_acc,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    **metrics_base
}

baseline_results_path = os.path.join(RESULTS_DIR, "baseline_results.csv")
pd.DataFrame([baseline_info]).to_csv(baseline_results_path, index=False)

Split: 16,000 train / 4,000 test
Training the model on baseline data...
   Baseline Accuracy: 0.7408 (74.08%)
   F1: 0.7407


### Corrupt the data -> Train and Evaluate -> Clean -> Re-train and evaluate

In [597]:
def run_batch_experiments_numerical(batch_id, corruption_list):
    """Run corruption, cleaning, and evaluation for a batch of experiments."""
    batch_results = []

    for corruption_idx, (exp_id, corrupt_func, columns) in enumerate(corruption_list, start=1):
        print(f"[{corruption_idx}/4] {exp_id}")

        # Corrupt baseline
        with timer() as t:
            X_corrupt, y_corrupt = corrupt_func(X_baseline.copy(), y_baseline.copy(), columns)
        corrupt_time = t
        n_before = len(X_corrupt)
        # Progressive cleaning levels
        cleaning_stages = [
            ("0_Corrupted", 0),
            ("1_Basic", 1),
            ("2_Heuristic", 2),
            ("3_Semantic", 3),
            ("4_ModelAware", 4),
        ]

        level_0_acc = None
        X_current = X_corrupt.copy()
        y_current = y_corrupt.copy()
        meta = {"levels_applied": [], "total_dropped": 0}
        for level_idx, (level_name, level_num) in enumerate(cleaning_stages):
            print(f"\n   >>> {level_name}")

            # Initialize timing/memory tracking
            clean_time = 0
            clean_mem_peak = 0
            train_time = 0
            train_mem_peak = 0

            # Apply cleaning (skip for Level 0)
            if level_idx > 0:
                print(f"       Cleaning (Level {level_num})")

                # START TRACKING CLEANING
                tracemalloc.start()
                clean_start = time.time()

                X_current, y_current, meta = clean.clean_progressive(
                    X_current, y_current, NUMERIC_FEATURES,
                    model_pipeline=build_model(),
                    levels=[level_num]
                )

                # END TRACKING CLEANING
                clean_time = time.time() - clean_start
                current_mem, peak_mem = tracemalloc.get_traced_memory()
                clean_mem_peak = peak_mem / 1024 / 1024  # Convert to MB
                tracemalloc.stop()

                print(f"Cleaning: {clean_time:.2f}s, Memory: {clean_mem_peak:.1f} MB")

                # Report cleaning stats
                if meta.get("total_dropped", 0) > 0:
                    print(f"Dropped: {meta['total_dropped']} rows")

            print(f"       Current size: {len(X_current):,} rows")

            # Evaluate
            print(f"       Training & evaluating")

            # START TRACKING TRAINING
            tracemalloc.start()
            train_start = time.time()
            with timer() as t:
                metrics, stats = train_and_evaluate(
                    X_current, X_current, y_current, y_current, train_ids, test_ids
                )
            eval_time = t

            #END TRACKING TRAINING
            train_time = time.time() - train_start
            current_mem, peak_mem = tracemalloc.get_traced_memory()
            train_mem_peak = peak_mem / 1024 / 1024  # Convert to MB
            tracemalloc.stop()

            print(f"Training: {train_time:.2f}s | Memory: {train_mem_peak:.1f} MB")

            if metrics is None:
                print(f"Skipped (insufficient data)")
                continue

            # Store level 0 accuracy for recovery calculation
            if level_idx == 0:
                level_0_acc = metrics['accuracy']

            # Calculate recovery
            if level_idx > 0 and level_0_acc is not None:
                acc_drop = baseline_acc - level_0_acc
                acc_recovery = metrics['accuracy'] - level_0_acc
                recovery_pct = (acc_recovery / acc_drop * 100) if acc_drop != 0 else 0
            else:
                recovery_pct = 0

            print(f"       Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")
            if level_idx > 0:
                print(f"       Recovery: {recovery_pct:.1f}%")

            # Record result
            result_row = {
               "batch": batch_id,
                "experiment": exp_id,
                "cleaning_name": level_name,
                "cleaning_num": level_num,
                **stats, **metrics,
                "baseline_acc": baseline_acc,
                "level_0_acc": level_0_acc or 0,
                "recovery_pct": recovery_pct,
                "corrupt_time": corrupt_time,
                "cleaning_time": clean_time,
                "eval_time": eval_time,
                "n_before": n_before,
                "n_after": len(X_current),
                **meta
            }
            batch_results.append(result_row)

        del X_corrupt, y_corrupt, X_current, y_current
        clear_memory()

    # Save and return
    batch_df = pd.DataFrame(batch_results)
    batch_path = os.path.join(RESULTS_DIR, f"{batch_id}_results.csv")
    batch_df.to_csv(batch_path, index=False)
    print(f"\n{batch_id} complete! Saved to {batch_path}")
    return batch_df

In [598]:
def run_batch_experiments_categorical(batch_id, corruption_list):
    """Run corruption, cleaning, and evaluation for a batch of experiments."""
    batch_results = []

    for corruption_idx, (exp_id, corrupt_func, columns) in enumerate(corruption_list, start=1):
        print(f"[{corruption_idx}/{len(corruption_list)}] {exp_id}")

        # Reset to baseline for each corruption

        # Corrupt
        with timer() as t:
            X_corrupted, y_corrupted = corrupt_func(X_baseline, y_baseline, CATEGORICAL_FEATURES)
        corrupt_time = t
        n_before = len(X_corrupted)
        # Test 3 cleaning levels for categorical:
        # Level 0: Corrupted (no cleaning)
        # Level 1: Drop corrupted
        # Level 2: Mode imputation
        cat_meta = {"levels_applied": [], "total_dropped": 0}
        for level_num in [0, 1, 2]:
            level_names = {0: "Corrupted", 1: "Basic", 2: "ModeImpute"}

            # Initialize timing/memory tracking
            clean_time = 0
            clean_mem_peak = 0
            train_time = 0
            train_mem_peak = 0

            # Start with corrupted data
            X_current = X_corrupted.copy()
            y_current = y_corrupted.copy()

            # Apply categorical cleaning if level > 0
            if level_num > 0:
                print(f"Cleaning (Level {level_num})")

                # START TRACKING CLEANING
                tracemalloc.start()
                clean_start = time.time()

                X_current, y_current, cat_meta = clean.clean_categorical_progressive(
                    X_current, y_current, CATEGORICAL_FEATURES, levels=[level_num]
                )

                # END TRACKING CLEANING
                clean_time = time.time() - clean_start
                current_mem, peak_mem = tracemalloc.get_traced_memory()
                clean_mem_peak = peak_mem / 1024 / 1024  # Convert to MB
                tracemalloc.stop()

                print(f"Cleaning: {clean_time:.2f}s, Memory: {clean_mem_peak:.1f} MB")

                if cat_meta['total_dropped'] > 0:
                    print(f"Dropped: {cat_meta['total_dropped']} rows")

            print(f"Current size: {len(X_current):,} rows")

            # Train/test split
            X_train, X_test, y_train, y_test = train_test_split(
                X_current, y_current, test_size=0.30, random_state=42
            )

            # Train and evaluate
            print(f"Training & evaluating")

            # START TRACKING TRAINING
            tracemalloc.start()
            train_start = time.time()
            with timer() as t:
                metrics, stats = train_and_evaluate(
                    X_current, X_current, y_current, y_current, train_ids, test_ids
                )
            eval_time = t
            model = build_model()

            # Handle any remaining NaN values
            X_train_clean = X_train.copy()
            X_test_clean = X_test.copy()

            # For categorical columns, fill NaN with mode
            for col in CATEGORICAL_FEATURES:
                if col in X_train_clean.columns:
                    mode_val = X_train_clean[col].mode()[0] if len(X_train_clean[col].mode()) > 0 else 'Unknown'
                    X_train_clean[col] = X_train_clean[col].fillna(mode_val)
                    X_test_clean[col] = X_test_clean[col].fillna(mode_val)

            model.fit(X_train_clean, y_train)
            y_pred = model.predict(X_test_clean)

            #END TRACKING TRAINING
            train_time = time.time() - train_start
            current_mem, peak_mem = tracemalloc.get_traced_memory()
            train_mem_peak = peak_mem / 1024 / 1024  # Convert to MB
            tracemalloc.stop()

            print(f"Training: {train_time:.2f}s, Memory: {train_mem_peak:.1f} MB")

            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

            print(f"Acc: {acc:.4f} | F1: {f1:.4f}")

            # Calculate recovery rate
            if level_num == 0:
                recovery = 0.0
            else:
                recovery = ((acc - level_0_acc) / (baseline_acc - level_0_acc) * 100) if (baseline_acc - level_0_acc) > 0 else 0.0
                print(f"Recovery: {recovery:.1f}%")

            # Store level 0 accuracy for recovery calculation
            if level_num == 0:
                level_0_acc = acc
            

            # Save result
            batch_results.append({
               "batch": batch_id,
                "experiment": exp_id,
                "cleaning_name": level_names[level_num],
                "cleaning_num": level_num,
                **stats, **metrics,
                "baseline_acc": baseline_acc,
                "level_0_acc": level_0_acc or 0,
                "recovery_pct": recovery,
                "corrupt_time": corrupt_time,
                "cleaning_time": clean_time,
                "eval_time": eval_time,
                "n_before": n_before,
                "n_after": len(X_current),
                **cat_meta
            })

    # Save and return
    batch_df = pd.DataFrame(batch_results)
    batch_path = os.path.join(RESULTS_DIR, f"{batch_id}_results.csv")
    batch_df.to_csv(batch_path, index=False)
    print(f"\n{batch_id} complete! Saved to {batch_path}")
    return batch_df

In [599]:
# Batch 1
batch1_corruptions = [
    ("01_all_numerical", corrupt.NUMERICAL_CORRUPTIONS["01_all_numerical"], NUMERIC_FEATURES),
    ("02_missing_values", corrupt.NUMERICAL_CORRUPTIONS["02_missing_values"], NUMERIC_FEATURES),
    ("03_scaling", corrupt.NUMERICAL_CORRUPTIONS["03_scaling"], NUMERIC_FEATURES),
    ("04_negation", corrupt.NUMERICAL_CORRUPTIONS["04_negation"], NUMERIC_FEATURES),
]

batch1_df = run_batch_experiments_numerical("batch1", batch1_corruptions)

[1/4] 01_all_numerical

   >>> 0_Corrupted
       Current size: 20,000 rows
       Training & evaluating
Training: 4.66s | Memory: 9.6 MB
       Acc: 0.5995 | F1: 0.5747

   >>> 1_Basic
       Cleaning (Level 1)
Cleaning: 0.99s, Memory: 7.2 MB
       Current size: 20,000 rows
       Training & evaluating
Training: 2.40s | Memory: 9.6 MB
       Acc: 0.6205 | F1: 0.6117
       Recovery: 14.9%

   >>> 2_Heuristic
       Cleaning (Level 2)
Cleaning: 0.04s, Memory: 5.2 MB
Dropped: 9347 rows
       Current size: 10,653 rows
       Training & evaluating
Training: 1.26s | Memory: 5.1 MB
       Acc: 0.6462 | F1: 0.6486
       Recovery: 33.1%

   >>> 3_Semantic
       Cleaning (Level 3)
Cleaning: 0.00s, Memory: 2.1 MB
       Current size: 10,653 rows
       Training & evaluating
Training: 1.13s | Memory: 5.1 MB
       Acc: 0.6462 | F1: 0.6486
       Recovery: 33.1%

   >>> 4_ModelAware
       Cleaning (Level 4)
Cleaning: 0.06s, Memory: 3.4 MB
       Current size: 10,653 rows
       Training & ev

In [600]:
# Batch 2

batch2_corruptions = [
    ("05_char_injection", corrupt.NUMERICAL_CORRUPTIONS["05_char_injection"], NUMERIC_FEATURES),
    ("06_combined_missing_scaling", corrupt.NUMERICAL_CORRUPTIONS["06_combined_missing_scaling"], NUMERIC_FEATURES),
    ("07_combined_negation_chars", corrupt.NUMERICAL_CORRUPTIONS["07_combined_negation_chars"], NUMERIC_FEATURES),
    ("08_heavy_missing", corrupt.NUMERICAL_CORRUPTIONS["08_heavy_missing"], NUMERIC_FEATURES),
    ("09_all_light", corrupt.NUMERICAL_CORRUPTIONS["09_all_light"], NUMERIC_FEATURES),
]

batch2_df = run_batch_experiments_numerical("batch2", batch2_corruptions)

[1/4] 05_char_injection

   >>> 0_Corrupted
       Current size: 20,000 rows
       Training & evaluating
Training: 2.52s | Memory: 9.6 MB
       Acc: 0.6740 | F1: 0.6732

   >>> 1_Basic
       Cleaning (Level 1)
Cleaning: 0.72s, Memory: 7.1 MB
       Current size: 20,000 rows
       Training & evaluating
Training: 1.88s | Memory: 9.6 MB
       Acc: 0.7408 | F1: 0.7407
       Recovery: 100.0%

   >>> 2_Heuristic
       Cleaning (Level 2)
Cleaning: 0.03s, Memory: 5.7 MB
Dropped: 4372 rows
       Current size: 15,628 rows
       Training & evaluating
Training: 1.59s | Memory: 7.5 MB
       Acc: 0.7164 | F1: 0.7180
       Recovery: 63.5%

   >>> 3_Semantic
       Cleaning (Level 3)
Cleaning: 0.00s, Memory: 3.1 MB
       Current size: 15,628 rows
       Training & evaluating
Training: 1.60s | Memory: 7.5 MB
       Acc: 0.7164 | F1: 0.7180
       Recovery: 63.5%

   >>> 4_ModelAware
       Cleaning (Level 4)
Cleaning: 0.08s, Memory: 5.0 MB
       Current size: 15,628 rows
       Training & 

In [601]:
# batch 3 (categorical corruptions)
batch3_corruptions = [
    ("10_category_shift", corrupt.apply_category_shift, CATEGORICAL_FEATURES),
    ("11_category_typo", corrupt.apply_category_typo, CATEGORICAL_FEATURES),
    ("12_category_default", corrupt.apply_category_default, CATEGORICAL_FEATURES),
    ("13_combined_categorical", corrupt.apply_combined_categorical, CATEGORICAL_FEATURES),
]

batch3_df = run_batch_experiments_categorical("batch3", batch3_corruptions)

[1/4] 10_category_shift
Current size: 20,000 rows
Training & evaluating
Training: 2.12s, Memory: 12.4 MB
Acc: 0.8030 | F1: 0.8023
Cleaning (Level 1)
Cleaning: 0.02s, Memory: 6.2 MB
Dropped: 9 rows
Current size: 19,991 rows
Training & evaluating
Training: 1.94s, Memory: 12.1 MB
Acc: 0.8184 | F1: 0.8184
Recovery: 0.0%
Cleaning (Level 2)
Cleaning: 0.01s, Memory: 4.8 MB
Current size: 20,000 rows
Training & evaluating
Training: 2.12s, Memory: 12.1 MB
Acc: 0.8028 | F1: 0.8021
Recovery: 0.0%
[2/4] 11_category_typo
Current size: 20,000 rows
Training & evaluating
Training: 2.06s, Memory: 12.4 MB
Acc: 0.6615 | F1: 0.6243
Cleaning (Level 1)
Cleaning: 0.06s, Memory: 5.5 MB
Dropped: 5817 rows
Current size: 14,183 rows
Training & evaluating
Training: 1.97s, Memory: 18.2 MB
Acc: 0.8188 | F1: 0.8187
Recovery: 198.5%
Cleaning (Level 2)
Cleaning: 0.02s, Memory: 4.8 MB
Current size: 20,000 rows
Training & evaluating
Training: 2.07s, Memory: 10.9 MB
Acc: 0.6488 | F1: 0.6478
Recovery: -16.0%
[3/4] 12_categ

### Combining batch1, batch2, and batch3 results

In [607]:
batch1_path = os.path.join(RESULTS_DIR, "batch1_results.csv")
batch1_df = pd.read_csv(batch1_path)
batch2_path = os.path.join(RESULTS_DIR, "batch2_results.csv")
batch2_df = pd.read_csv(batch2_path)
batch3_path = os.path.join(RESULTS_DIR, "batch3_results.csv")
batch3_df = pd.read_csv(batch3_path)
all_results_df = pd.concat([batch1_df, batch2_df, batch3_df], ignore_index=True)
combined_results_path = os.path.join(RESULTS_DIR, "all_results_combined.csv")
all_results_df.to_csv(combined_results_path, index=False)

print(f"Combined results and saved to: {combined_results_path}")
print(f"Total evaluations: {len(all_results_df)}")

Combined results and saved to: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\results/all_results_combined.csv
Total evaluations: 57


## Analysis of Results


In [608]:
results_df = pd.read_csv(combined_results_path)
print(f"Loaded results DF with {len(results_df)} rows for analysis")

Loaded results DF with 57 rows for analysis


In [ ]:
# Constants for analysis
PERFORMANCE_DIR = os.path.join(FIGURES_DIR, "performance_analysis/")
os.makedirs(PERFORMANCE_DIR, exist_ok=True)

COMP_DIR = os.path.join(FIGURES_DIR, "computation_analysis/")
os.makedirs(COMP_DIR, exist_ok=True)

CLEANLAB_DIR = os.path.join(FIGURES_DIR, "cleanlab_analysis/")
os.makedirs(CLEANLAB_DIR, exist_ok=True)

BASELINE_ACC = results_df['baseline_acc'].iloc[0]
CLEANING_STRATEGIES_ORDER = ['1_Basic', '2_Heuristic', '3_Semantic', '4_ModelAware', "Corrupted", "Basic", "ModeImpute"]
ALL_CLEANING_ORDER = ['0_Corrupted'] + CLEANING_STRATEGIES_ORDER
HEATMAP_COLS = ['0_Corrupted'] + CLEANING_STRATEGIES_ORDER
PLOT_STYLE = {
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9
}
sns.set_style("whitegrid")
plt.rcParams.update(PLOT_STYLE)

In [610]:
# Helper function for plot saving
def save_plot(plot_type, plot_title, filename, **kwargs):
    """Helper to create, save, and close plots"""
    print(f"Generating plot: {plot_title}")
    plt.tight_layout()
    full_path = os.path.join(PERFORMANCE_DIR if plot_type == "performance" else COMP_DIR if plot_type == "computation" else CLEANLAB_DIR, filename)
    plt.savefig(full_path, **kwargs)
    plt.close()
    print(f"Saved: {full_path}")

### Corruption Impact Analysis on Performance

In [611]:
df = results_df.copy()

def interpret_damage(d):
    abs_d = abs(d)
    if abs_d < 0.03: return "Minimal"
    elif abs_d < 0.08: return "Moderate"
    elif abs_d < 0.15: return "Severe"
    else: return "Critical"

print("CORRUPTION IMPACT ANALYSIS (Baseline vs. Corrupted)")

corruption_df = df[df['cleaning_name'] == '0_Corrupted'].copy()

corruption_df['Damage'] = corruption_df['baseline_acc'] - corruption_df['accuracy']
corruption_df['Damage_Pct'] = (corruption_df['Damage'] / corruption_df['baseline_acc']) * 100
corruption_df['Damage_Level'] = corruption_df['Damage'].apply(interpret_damage)

report = corruption_df[[
    'experiment', 'baseline_acc', 'accuracy', 'Damage', 'Damage_Pct', 'Damage_Level'
]].copy()

report.columns = ['Experiment', 'Original_Acc', 'Corrupted_Acc', 'Abs_Damage', 'Damage_%', 'Severity']

# sort by damage
report = report.sort_values(by='Abs_Damage', ascending=False)

print(report.to_string(index=False))

# Key Findings
print("\nKEY FINDINGS: NOISE SENSITIVITY")

# exclude missing_labels because that causes a crash
most_dmg = report[report['Experiment'] != '04_missing_labels'].iloc[0]
least_dmg = report[report['Experiment'] != '04_missing_labels'].iloc[-1]

print(f"Most Destructive Corruption: {most_dmg['Experiment']}")
print(f"Accuracy dropped by {most_dmg['Damage_%']:.1f}% ({most_dmg['Severity']})")

print(f"\nMost Resilient Corruption:  {least_dmg['Experiment']}")
print(f"Accuracy dropped by {least_dmg['Damage_%']:.1f}% ({least_dmg['Severity']})")

avg_dmg = report['Abs_Damage'].mean()
print(f"\nAverage Accuracy Drop across all 4 types: {avg_dmg:.4f}")

CORRUPTION IMPACT ANALYSIS (Baseline vs. Corrupted)
                 Experiment  Original_Acc  Corrupted_Acc  Abs_Damage  Damage_% Severity
           01_all_numerical       0.74075        0.59950     0.14125 19.068512   Severe
               09_all_light       0.74075        0.61100     0.12975 17.516031   Severe
                04_negation       0.74075        0.61250     0.12825 17.313534   Severe
06_combined_missing_scaling       0.74075        0.61250     0.12825 17.313534   Severe
           08_heavy_missing       0.74075        0.62375     0.11700 15.794803   Severe
 07_combined_negation_chars       0.74075        0.63125     0.10950 14.782315   Severe
                 03_scaling       0.74075        0.63675     0.10400 14.039825   Severe
          02_missing_values       0.74075        0.65425     0.08650 11.677354   Severe
          05_char_injection       0.74075        0.67400     0.06675  9.011137 Moderate

KEY FINDINGS: NOISE SENSITIVITY
Most Destructive Corruption: 01_all

### Statitical Tests

In [612]:
STATS_DIR = os.path.join(RESULTS_DIR, "stats_analysis")
os.makedirs(STATS_DIR, exist_ok=True)

analysis_df = results_df.copy()

print("STATISTICAL ANALYSIS - INDEPENDENT METHOD EVALUATION")

experiments = analysis_df['experiment'].unique()
# ANALYSIS 1: STRATEGY PERFORMANCE (Treatment vs Control)
# tests every cleaning method against the corrupted baseline

stats_results = []

for exp in experiments:
    exp_data = analysis_df[analysis_df['experiment'] == exp]

    corrupted = exp_data[exp_data['cleaning_name'] == '0_Corrupted']
    if corrupted.empty: continue

    acc_corrupt = corrupted['accuracy'].values[0]
    damage = BASELINE_ACC - acc_corrupt

    for strategy in CLEANING_STRATEGIES_ORDER:
        method_data = exp_data[exp_data['cleaning_name'] == strategy]
        if method_data.empty: continue

        acc_clean = method_data['accuracy'].values[0]
        n_test = method_data['test_valid'].values[0]

        recovery_abs = acc_clean - acc_corrupt
        recovery_pct = method_data['recovery_pct'].values[0]

        data_loss = ((method_data['n_before'].values[0] - method_data['n_after'].values[0])
                     / method_data['n_before'].values[0] * 100)

        # z-test
        se = np.sqrt((acc_corrupt * (1 - acc_corrupt) + acc_clean * (1 - acc_clean)) / n_test)
        if se > 0:
            z_score = recovery_abs / se
            p_value = 2 * (1 - norm.cdf(abs(z_score)))
        else:
            p_value = 1.0

        sig = "***" if p_value < 0.001 else ("**" if p_value < 0.01 else ("*" if p_value < 0.05 else "ns"))

        stats_results.append({
            'Experiment': exp,
            'Strategy': strategy,
            'Baseline': BASELINE_ACC,
            'Corrupted': acc_corrupt,
            'Cleaned': acc_clean,
            'Abs_Gain': recovery_abs,
            'Recovery_Pct': recovery_pct,
            'Data_Loss': f"{data_loss:.1f}%",
            'p_value': p_value,
            'Significance': sig,
            'Damage_Level': "Significant" if damage > 0.05 else "Minor"
        })

df_stats = pd.DataFrame(stats_results)
stats_path = os.path.join(STATS_DIR, "method_wise_statistical_analysis.csv")
df_stats.to_csv(stats_path, index=False)

# ANALYSIS 2: METHOD COMPARISON MATRIX
# shows which method is best per corruption

print("\nCross-Method Accuracy Comparison:")
pivot_acc = analysis_df.pivot(index='experiment', columns='cleaning_name', values='accuracy')
print(pivot_acc.to_string())

# Summary
print("SUMMARY FINDINGS")

# avg effectiveness of each strategy
avg_rec = df_stats.groupby('Strategy')['Recovery_Pct'].mean()
print("\n  Average Recovery % by Strategy:")
for strat, val in avg_rec.items():
    print(f"   {strat:15s}: {val:.2f}%")

# significant wins
sig_wins = df_stats[df_stats['p_value'] < 0.05].groupby('Strategy').size()
print("\nSignificant Improvements (p<0.05) count:")
print(sig_wins if not sig_wins.empty else "No strategies reached statistical significance.")

# best Strategy for each corruption
print("\nBest Strategy per Experiment:")
idx_best = df_stats.groupby('Experiment')['Cleaned'].idxmax()
best_df = df_stats.loc[idx_best, ['Experiment', 'Strategy', 'Cleaned', 'Recovery_Pct', 'Significance']]
print(best_df.to_string(index=False))

del analysis_df

STATISTICAL ANALYSIS - INDEPENDENT METHOD EVALUATION

Cross-Method Accuracy Comparison:
cleaning_name                0_Corrupted  1_Basic  2_Heuristic  3_Semantic  4_ModelAware     Basic  Corrupted  ModeImpute
experiment                                                                                                               
01_all_numerical                 0.59950  0.62050     0.646197    0.646197      0.646197       NaN        NaN         NaN
02_missing_values                0.65425  0.72175     0.705521    0.705521      0.705521       NaN        NaN         NaN
03_scaling                       0.63675  0.63675     0.675227    0.675227      0.675227       NaN        NaN         NaN
04_negation                      0.61250  0.74075     0.716399    0.716399      0.716399       NaN        NaN         NaN
05_char_injection                0.67400  0.74075     0.716399    0.716399      0.716399       NaN        NaN         NaN
06_combined_missing_scaling      0.61250  0.61400     0.65

### Visual Analysis of Performance

In [613]:
PERFORMANCE_DIR = os.path.join(FIGURES_DIR, "performance_analysis")
os.makedirs(PERFORMANCE_DIR, exist_ok=True)

# styles
plt.rcParams.update(PLOT_STYLE)

analysis_df = results_df.copy()

analysis_df['data_loss_pct'] = ((analysis_df['n_before'] - analysis_df['n_after']) / analysis_df['n_before']) * 100
analysis_df['accuracy_gain'] = analysis_df['accuracy'] - analysis_df['level_0_acc']

# FIGURE 1: Corruption Damage Ranking (Bar Chart)
corruption_data = analysis_df[analysis_df['cleaning_name'] == '0_Corrupted'].copy()
corruption_data['Damage'] = corruption_data['baseline_acc'] - corruption_data['accuracy']
corruption_data = corruption_data.sort_values('accuracy', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(corruption_data['experiment'], corruption_data['Damage'],
        color='salmon', edgecolor='black', alpha=0.8)

ax.set_xlabel('Accuracy Drop (Baseline - Corrupted)')
ax.set_title('Corruption Impact: Accuracy Degradation by Type')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
save_plot("performance", "Corruption Damage Ranking", "01_corruption_damage_ranking.png")

#FIGURE 2: Average Model Accuracy in increasing order (Bar Chart)
avg_acc = analysis_df[analysis_df['cleaning_name'] != '0_Corrupted'].groupby('cleaning_name')['accuracy'].mean().reset_index()
avg_corruption_acc = analysis_df[analysis_df['cleaning_name'] == '0_Corrupted']['accuracy'].mean()
avg_acc = avg_acc.sort_values('accuracy', ascending=True)

colors = plt.cm.viridis(np.linspace(0, 0.8, len(avg_acc)))

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.bar(avg_acc['cleaning_name'], avg_acc['accuracy'],
              color=colors, edgecolor='black', alpha=0.85)
ax.axhline(y=avg_corruption_acc, color='red', linestyle='--', linewidth=2.5,
           label=f'Avg Corrupted Acc ({avg_corruption_acc:.3f})', zorder=3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
            f'{height:.3f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold', color='black')

ax.set_ylim(0.40, 0.52)
import matplotlib.ticker as ticker
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.0025))

ax.grid(which='major', axis='y', linestyle='-', alpha=0.3, color='gray')
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.2, color='gray')
ax.set_axisbelow(True)

ax.set_xlabel('Cleaning Strategy', fontsize=12, labelpad=10)
ax.set_ylabel('Average Model Accuracy', fontsize=12, labelpad=10)
ax.set_title('Detailed Strategy Effectiveness vs. Baseline', fontsize=14, fontweight='bold', pad=20)

ax.legend(frameon=True, shadow=True, loc='upper left')
save_plot("performance", "Average Model Accuracy", "02_average_model_accuracy.png")

# FIGURE 3: Strategy Comparison Heatmap
df_heatmap = analysis_df.pivot(index='experiment', columns='cleaning_name', values='accuracy')
df_heatmap = df_heatmap[HEATMAP_COLS]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_heatmap, annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Strategy Comparison: Accuracy Across Cleaning Methods')
ax.set_xlabel('Cleaning Strategy')
ax.set_ylabel('Corruption Type')
save_plot("performance", "Strategy Comparison Heatmap", "03_strategy_comparison_heatmap.png")

# FIGURE 4: Best Strategy Performance Ranking
best_per_exp = analysis_df.loc[analysis_df.groupby('experiment')['accuracy'].idxmax()].copy()
best_per_exp = best_per_exp.sort_values('accuracy', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
unique_strats = best_per_exp['cleaning_name'].unique()
colors = plt.get_cmap('Set2')(range(len(unique_strats)))
strat_color_map = {strat: colors[i] for i, strat in enumerate(unique_strats)}

ax.barh(best_per_exp['experiment'], best_per_exp['accuracy'],
        color=[strat_color_map[s] for s in best_per_exp['cleaning_name']],
        edgecolor='black', alpha=0.8)

ax.axvline(x=BASELINE_ACC, color='blue', linestyle='--', linewidth=2, label=f'Baseline ({BASELINE_ACC:.2f})')
ax.set_xlabel('Highest Accuracy Achieved')
ax.set_title('Final Model Performance: Best Result per Corruption')
ax.set_xlim(analysis_df['accuracy'].min() * 0.9, BASELINE_ACC * 1.1)

legend_elements = [Patch(facecolor=strat_color_map[s], label=s) for s in unique_strats]
legend_elements.append(plt.Line2D([0], [0], color='blue', linestyle='--', label='Baseline'))
ax.legend(handles=legend_elements, loc='lower right')

save_plot("performance", "Best Strategy Performance Ranking", "04_best_strategy_performance_ranking.png")

# FIGURE 5: Trade-off: Data Loss vs. Cleaning Effectiveness
plt.figure(figsize=(10, 7))
df_plot = analysis_df[analysis_df['cleaning_name'] != '0_Corrupted']
sns.scatterplot(data=df_plot, x='data_loss_pct', y="accuracy", hue='cleaning_name', style='cleaning_name', s=100)

plt.axhline(y=BASELINE_ACC, color='blue', linestyle='--', alpha=0.5, label='Baseline')
plt.ylabel('Model Accuracy')

plt.title("Trade-off: Data Loss vs. Cleaning Effectiveness", fontweight='bold')
plt.xlabel('Data Loss (%)')
plt.legend(title='Strategy', bbox_to_anchor=(1.05, 1), loc='upper left')
save_plot("performance", "Trade-off Data Loss vs Cleaning Effectiveness", "05_data_loss_vs_effectiveness.png")

# FIGURE 6: Accuracy Gain Heatmap
pivot_acc = analysis_df.pivot(index='experiment', columns='cleaning_name', values='accuracy')[HEATMAP_COLS]
delta_df = (pivot_acc.subtract(pivot_acc['0_Corrupted'], axis=0).divide(pivot_acc['0_Corrupted'], axis=0) * 100).drop(columns='0_Corrupted')

plt.figure(figsize=(12, 8))
sns.heatmap(delta_df, annot=True, fmt='.3f', cmap='YlGn', linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Absolute Accuracy Gain (Points)'})
plt.title('Strategy Effectiveness: Accuracy Gain Over Corrupted Baseline', fontsize=14, fontweight='bold', pad=20)
save_plot("performance", "Accuracy Gain Heatmap", "06_accuracy_gain_heatmap.png")

# FIGURE 7: Recovery Robustness
df_strategies = analysis_df[analysis_df['cleaning_name'].isin(CLEANING_STRATEGIES_ORDER)].copy()
df_strategies['cleaning_name'] = pd.Categorical(df_strategies['cleaning_name'], categories=CLEANING_STRATEGIES_ORDER, ordered=True)

df_strategies['recovery_pct_viz'] = df_strategies['recovery_pct'].clip(lower=-20, upper=150)
plt.figure(figsize=(10, 6))
sns.boxplot(x='cleaning_name', y='recovery_pct_viz', data=df_strategies, palette='viridis', showmeans=True,
            meanprops={"marker":"s","markerfacecolor":"white", "markeredgecolor":"black"})
sns.stripplot(x='cleaning_name', y='recovery_pct_viz', data=df_strategies, color=".3", alpha=0.6, size=6)
plt.axhline(y=100, color='red', linestyle='--', label='100% Recovery')
plt.title('Strategy Recovery Success (Capped at 150%)', fontweight='bold')
plt.legend()
save_plot("performance", "Recovery Robustness", "07_recovery_robustness_boxplot.png")

# FIGURE 8: Absolute Accuracy Boxplot
plt.figure(figsize=(10, 6))
all_order = ['0_Corrupted'] + CLEANING_STRATEGIES_ORDER
plot_all_df = analysis_df[analysis_df['cleaning_name'].isin(all_order)].copy()
plot_all_df['cleaning_name'] = pd.Categorical(plot_all_df['cleaning_name'], categories=all_order, ordered=True)

sns.boxplot(x='cleaning_name', y='accuracy', data=plot_all_df, palette='Set2', showmeans=True)
sns.swarmplot(x='cleaning_name', y='accuracy', data=plot_all_df, color=".25", alpha=0.6)
plt.axhline(y=BASELINE_ACC, color='blue', linestyle='--', linewidth=2, label=f'Baseline ({BASELINE_ACC:.3f})')
plt.title('Performance Distribution: Absolute Accuracy', fontsize=14, fontweight='bold')
plt.legend()
save_plot("performance", "Accuracy Distribution", "08_accuracy_distribution_boxplot.png")

# FIGURE 9: Impact Bar Chart
df_corrupted = analysis_df[analysis_df['cleaning_name'] == '0_Corrupted'].copy()
df_corrupted['Drop'] = df_corrupted['baseline_acc'] - df_corrupted['accuracy']
df_corrupted = df_corrupted.sort_values('Drop', ascending=False)

melted_df = df_corrupted.melt(id_vars=['experiment', 'Drop'], value_vars=['baseline_acc', 'accuracy'],
                              var_name='Metrics', value_name='Accuracy')

plt.figure(figsize=(12, 7))
ax = sns.barplot(data=melted_df, x='experiment', y='Accuracy', hue='Metrics', palette=['#3498db', '#e74c3c'])
for i, (_, row) in enumerate(df_corrupted.iterrows()):
    ax.text(i + 0.2, row['accuracy'] + 0.01, f"-{row['Drop']:.1%}",
            ha='center', va='bottom', color='red', fontweight='bold', fontsize=9)

plt.xticks(rotation=45, ha='right')
plt.title('Impact of Corruption: Baseline vs. Corrupted', fontweight='bold')
save_plot("performance", "Impact of Corruption", "09_corruption_impact_bar_chart.png")

# FIGURE 10: Recovery Trends
severity_order = df_corrupted.sort_values('Drop', ascending=False)['experiment'].unique().tolist()

df_strat_trend = analysis_df[analysis_df['cleaning_name'].isin(CLEANING_STRATEGIES_ORDER)].copy()
df_strat_trend['cleaning_name'] = pd.Categorical(df_strat_trend['cleaning_name'], categories=CLEANING_STRATEGIES_ORDER, ordered=True)
df_strat_trend['experiment'] = pd.Categorical(df_strat_trend['experiment'], categories=severity_order, ordered=True)
df_strat_trend['recovery_percent'] = df_strat_trend['recovery_pct'].clip(lower=-20, upper=120)

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_strat_trend, x='experiment', y='recovery_percent', hue='cleaning_name',
             marker='o', markersize=8, linewidth=2, palette='viridis')

plt.axhline(y=100, color='red', linestyle='--', alpha=0.6, label='Full Recovery (100%)')
plt.ylim(-30, 130)
plt.xticks(rotation=45, ha='right')
plt.title('Performance Recovery Trends Across Corruption Severity', fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
save_plot("performance", "Performance Recovery Trends", "10_performance_recovery_trends.png")

print(f"All performance analysis figures saved to: {PERFORMANCE_DIR}")

del analysis_df, corruption_data, avg_acc, df_heatmap, best_per_exp, df_plot, delta_df, df_strategies, plot_all_df, df_corrupted, melted_df, df_strat_trend
clear_memory()

Generating plot: Corruption Damage Ranking
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/performance_analysis\01_corruption_damage_ranking.png
Generating plot: Average Model Accuracy
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/performance_analysis\02_average_model_accuracy.png
Generating plot: Strategy Comparison Heatmap
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/performance_analysis\03_strategy_comparison_heatmap.png
Generating plot: Best Strategy Performance Ranking
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/performance_analysis\04_best_strategy_performance_ranking.png
Generating plot: Trade-off Data Loss vs Cleaning Effectiveness
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/performance_ana

### Visual Analysis of Computation

In [ ]:
from matplotlib.ticker import ScalarFormatter

COMP_DIR = os.path.join(FIGURES_DIR, "computation_analysis")
os.makedirs(COMP_DIR, exist_ok=True)

time_cols = ['corrupt_time', 'cleaning_time', 'eval_time']
analysis_df = results_df.copy()
for col in time_cols:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors='coerce')

# extra metrics
analysis_df['total_time'] = analysis_df['corrupt_time'] + analysis_df['cleaning_time'] + analysis_df['eval_time']
analysis_df['cleaning_ratio'] = analysis_df['cleaning_time'] / analysis_df['total_time']
analysis_df['eval_ratio'] = analysis_df['eval_time'] / analysis_df['total_time']
analysis_df['corrupt_ratio'] = analysis_df['corrupt_time'] / analysis_df['total_time']

analysis_df['cleaning_name'] = pd.Categorical(analysis_df['cleaning_name'], categories=ALL_CLEANING_ORDER, ordered=True)

# FIGURE 1: Time Breakdown Stacked Bar Chart
avg_time = analysis_df.groupby('cleaning_name')[['cleaning_time', 'eval_time']].mean()

ax = avg_time.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#3498db', '#2ecc71'])
plt.title('Average Computational Time Breakdown per Strategy', fontweight='bold')
plt.ylabel('Time (Seconds)')
plt.xlabel('Cleaning Strategy')
plt.xticks(rotation=0)
plt.legend(['Cleaning Time', 'Eval Time'])
save_plot("computation", "Time Breakdown Stacked Bar Chart", "01_time_breakdown_stacked_bar.png")

# FIGURE 2: Total Pipeline Time Comparision
plt.figure(figsize=(10, 6))
sns.barplot(x='cleaning_name', y='total_time', data=analysis_df, palette='magma', errorbar=None)
plt.title('Total Pipeline Runtime Comparison (End-to-End)', fontweight='bold')
plt.ylabel('Total Time (Seconds)')
plt.xlabel('Cleaning Strategy')
save_plot("computation", "Total Pipeline Time Comparison", "02_total_pipeline_time.png")

# FIGURE 3: Boxplot of Time Variability
df_summary = analysis_df.groupby('cleaning_name').agg({
    'accuracy': 'mean',
    'total_time': 'mean'
}).reset_index()
df_summary = df_summary[df_summary['cleaning_name'] != '0_Corrupted'].copy()
df_summary['label'] = df_summary['cleaning_name']

df_summary = df_summary.sort_values('total_time')

plt.figure(figsize=(10, 6))
sns.set_style({'axes.grid': True, 'grid.linestyle': '--'})

plt.xscale('log')
plt.plot(df_summary['total_time'], df_summary['accuracy'],
         color='lightgrey', linestyle='--', linewidth=1.5, zorder=1)
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']
for i, (idx, row) in enumerate(df_summary.iterrows()):
    plt.scatter(row['total_time'], row['accuracy'], s=500,
                color=colors[i % len(colors)], edgecolor='black',
                alpha=0.9, linewidth=1.5, zorder=2)

    x_offset = 1.05
    y_offset = 0.0008

    plt.text(row['total_time'] * x_offset, row['accuracy'] + y_offset,
             row['label'], fontweight='bold', fontsize=11,
             verticalalignment='center')

plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.xticks([100, 150, 200, 300, 400, 500])

plt.title('Accuracy vs. Time Trade-off', fontsize=16, fontweight='bold', pad=25)
plt.xlabel('Total Computational Time (Seconds)', fontsize=12)
plt.ylabel('Average Model Accuracy', fontsize=12)
sns.despine(left=True, bottom=True)
plt.grid(True, which="both", ls="--", alpha=0.4)
save_plot("computation", "Accuracy vs Time Trade-off", "03_accuracy_vs_time_tradeoff.png")

# FIGURE 4: Cleaning Overhead Ratio
ratio_data = analysis_df.groupby('cleaning_name')[['corrupt_ratio', 'cleaning_ratio', 'eval_ratio']].mean()

ax = ratio_data.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#95a5a6', '#3498db', '#2ecc71'])
plt.title('Computational Overhead Ratio: Where is the bottleneck?', fontweight='bold')
plt.ylabel('Percentage of Total Runtime')
plt.xlabel('Cleaning Strategy')
plt.xticks(rotation=0)
plt.legend(['Corrupt %', 'Cleaning %', 'Eval %'], loc='lower right')

vals = ax.get_yticks()
ax.set_yticklabels(['{:,.0%}'.format(x) for x in vals])

save_plot("computation", "Cleaning Overhead Ratios", "04_cleaning_overhead_ratios.png")

# FIGURE 5: Corruption-specific Time Sensitivity
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=analysis_df,
    x='experiment',
    y='cleaning_time',
    hue='cleaning_name',
    marker='o',
    markersize=8,
    linewidth=2,
    sort=False
)

plt.yscale('log')
plt.title(
    'Cleaning Time Sensitivity across Corruption Types (Log Scale)',
    fontweight='bold'
)
plt.ylabel('Cleaning Time (seconds, log scale)')
plt.xlabel('Corruption Experiment')
plt.xticks(rotation=45, ha='right')

plt.yticks([10, 100])
plt.ylim(0.8, analysis_df['cleaning_time'].max() * 1.1)

plt.legend(title='Strategy', loc='upper right')

plt.tight_layout()
save_plot("computation", "Time Sensitivity per Corruption", "05_time_sensitivity_per_corruption.png")

print("\nAll computation analysis figures saved to the 'computation_analysis' folder.")

del analysis_df, avg_time, ratio_data, df_summary
clear_memory()

Generating plot: Time Breakdown Stacked Bar Chart
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/computation_analysis\01_time_breakdown_stacked_bar.png
Generating plot: Total Pipeline Time Comparison
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/computation_analysis\02_total_pipeline_time.png
Generating plot: Accuracy vs Time Trade-off
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/computation_analysis\03_accuracy_vs_time_tradeoff.png
Generating plot: Cleaning Overhead Ratios
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/computation_analysis\04_cleaning_overhead_ratios.png
Generating plot: Time Sensitivity per Corruption
Saved: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\figures/computation_analysis\05_time_sensitiv

### Visual Analysis of Model Aware Reweighting using CleanLab

#### Combining batch1 and batch2 cleanlab details

In [616]:
batch1_cleanlab_path = os.path.join(RESULTS_DIR, "batch1_cleanlab_details.csv")
batch2_cleanlab_path = os.path.join(RESULTS_DIR, "batch2_cleanlab_details.csv")

df_cleanlab1 = pd.read_csv(batch1_cleanlab_path)
df_cleanlab2 = pd.read_csv(batch2_cleanlab_path)

df_cleanlab_combined = pd.concat([df_cleanlab1, df_cleanlab2], ignore_index=True)

combined_cleanlab_path = os.path.join(RESULTS_DIR, "all_cleanlab_stats_combined.csv")

df_cleanlab_combined.to_csv(combined_cleanlab_path, index=False)

print(f"Combined cleanlab details and saved to: {combined_cleanlab_path}")
print(f"Total cleanlab evaluations: {len(df_cleanlab_combined)}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\saite\\MastersStuff\\Data_Preperation\\Project\\Data_Preparation_Project_G17_Theme1\\results/batch1_cleanlab_details.csv'

#### Analysis and Visualization of CleanLab Results

In [617]:
CLEANLAB_DIR = os.path.join(FIGURES_DIR, "cleanlab_analysis")
os.makedirs(CLEANLAB_DIR, exist_ok=True)

# Figure 1: Confidence Separation
plt.figure(figsize=(8, 6))
conf_melted = df_cleanlab_combined.melt(id_vars=['experiment'], value_vars=['avg_conf_clean', 'avg_conf_noisy'],
                                 var_name='Confidence Type', value_name='Score')
conf_melted['Confidence Type'] = conf_melted['Confidence Type'].replace({
    'avg_conf_clean': 'Clean Samples',
    'avg_conf_noisy': 'Noisy/Issue Samples'
})

sns.boxplot(x='Confidence Type', y='Score', data=conf_melted, palette='Set2')
sns.stripplot(x='Confidence Type', y='Score', data=conf_melted, color=".3", alpha=0.5)
plt.title('Cleanlab Issue Detection: Predicted Confidence Separation', fontsize=14, fontweight='bold')
plt.ylabel('Mean Predicted Confidence')
plt.xlabel('')
save_plot("cleanlab", "Confidence Separation", "01_confidence_separation.png")

# Figure 2: Class-wise Noise Analysis
noise_cols = [f'class_noise_{i}' for i in range(1, 6)]
heatmap_data = df_cleanlab_combined.set_index('experiment')[noise_cols]
heatmap_data.columns = [f'Class {i}' for i in range(1, 6)]

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, cmap='YlOrRd', fmt='.3f', cbar_kws={'label': 'Estimated Noise Proportion'})
plt.title('Class-Specific Noise Estimated by Cleanlab', fontsize=14, fontweight='bold')
plt.ylabel('Corruption Experiment')
plt.xlabel('Review Rating (Class)')
save_plot("cleanlab", "Class-wise Noise Heatmap", "02_class_wise_noise_heatmap.png")

# Figure 3: Reweighting Behavior Analysis
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_cleanlab_combined, x='issue_rate', y='avg_weight', hue='experiment',
                s=150, palette='tab10', alpha=0.8)

z = np.polyfit(df_cleanlab_combined['issue_rate'], df_cleanlab_combined['avg_weight'], 1)
p = np.poly1d(z)
x_range = np.linspace(df_cleanlab_combined['issue_rate'].min(), df_cleanlab_combined['issue_rate'].max(), 100)
plt.plot(x_range, p(x_range), "r--", alpha=0.5, label='General Trend')

plt.title('Sample Reweighting vs. Detected Noise (Issue Rate)', fontsize=14, fontweight='bold')
plt.xlabel('Detected Issue Rate (proportion)')
plt.ylabel('Average Sample Weight')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Experiment')
plt.grid(True, linestyle='--', alpha=0.6)
save_plot("cleanlab", "Reweighting Behavior Analysis", "03_reweighting_behavior_analysis.png")

print(f"All CleanLab analysis figures saved to: {CLEANLAB_DIR}")

NameError: name 'df_cleanlab_combined' is not defined

<Figure size 2400x1800 with 0 Axes>

## Computing and Saving some Relevant Statistics

### Performance Related

In [623]:
dupes = (
    results_df.groupby(['experiment', 'cleaning_name'])
      .size()
      .reset_index(name='count')
      .query('count > 1')
      .sort_values('count', ascending=False)
)

dupes

,experiment,cleaning_name,count


In [630]:
df = results_df.copy()

time_cols = ['corrupt_time', 'cleaning_time', 'eval_time']
for col in time_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['total_time'] = df['corrupt_time'] + df['cleaning_time'] + df['eval_time']
df['cleaning_name'] = pd.Categorical(df['cleaning_name'], categories=ALL_CLEANING_ORDER, ordered=True)

dupes = (
    df.groupby(['experiment', 'cleaning_name'])
      .size()
      .reset_index(name='count')
      .query('count > 1')
      .sort_values('count', ascending=False)
)

print(df[['experiment', 'cleaning_name']].isna().sum())
df = df.dropna(subset=['experiment', 'cleaning_name'])


# Table 1: Corruption Severity (Baseline vs Corrupted)
print("Generating Table 1: Corruption Severity...")
t1 = df[df['cleaning_name'] == '0_Corrupted'][['experiment', 'baseline_acc', 'accuracy']].copy()
t1['accuracy_drop'] = t1['baseline_acc'] - t1['accuracy']
t1 = t1.rename(columns={'accuracy': 'corrupted_acc'})
t1 = t1.sort_values('accuracy_drop', ascending=False).round(3)
file_name = os.path.join(STATS_DIR, 'table_01_corruption_severity.csv')
t1.to_csv(file_name, index=False)

# Table 2: Average Cleaning Method Effectiveness
print("Generating Table 2: Method Effectiveness...")
t2 = df.groupby('cleaning_name')[['accuracy', 'f1', 'recovery_pct']].mean().round(3)
file_name = os.path.join(STATS_DIR, 'table_02_method_effectiveness.csv')
t2.to_csv(file_name)

# 3. Table 3: Cleaning x Corruption Interaction
print("Generating Table 3: Interaction Analysis...")
t3_acc = df.pivot(index='experiment', columns='cleaning_name', values='accuracy')
t3_rec = df.pivot(index='experiment', columns='cleaning_name', values='recovery_pct')

# Flattening the pivot for a clean CSV structure
t3_acc.columns = [f"{col}_accuracy" for col in t3_acc.columns]
t3_rec.columns = [f"{col}_recovery" for col in t3_rec.columns]
t3 = pd.concat([t3_acc, t3_rec], axis=1).round(3)
file_name = os.path.join(STATS_DIR, 'table_03_cleaning_interaction.csv')
t3.to_csv(file_name)

# 4. Table 4: Computational Cost Summary (Computational Cost Summary)
print("Generating Table 4: Computational Cost...")
t4 = df.groupby('cleaning_name')[['corrupt_time', 'cleaning_time', 'eval_time', 'total_time']].mean().round(3)
file_name = os.path.join(STATS_DIR, 'table_04_computational_cost.csv')
t4.to_csv(file_name)

# 5. Table 5: Data Impact Summary
print("Generating Table 5: Data Impact Summary...")
t5 = df.groupby('cleaning_name')[['n_before', 'n_after']].mean()
t5['pct_data_affected'] = (100 * (t5['n_before'] - t5['n_after']) / t5['n_before']).round(2)
t5 = t5.round(0)
t5['n_before'] = t5['n_before'].astype(int)
t5['n_after'] = t5['n_after'].astype(int)
file_name = os.path.join(STATS_DIR, 'table_05_data_impact.csv')
t5.to_csv(file_name)

print(f"\nAnalysis Tables generated and saved to: {STATS_DIR}")

del df, t1, t2, t3, t3_acc, t3_rec, t4, t5
clear_memory()

experiment        0
cleaning_name    12
dtype: int64
Generating Table 1: Corruption Severity...
Generating Table 2: Method Effectiveness...
Generating Table 3: Interaction Analysis...
Generating Table 4: Computational Cost...
Generating Table 5: Data Impact Summary...

Analysis Tables generated and saved to: c:\Users\saite\MastersStuff\Data_Preperation\Project\Data_Preparation_Project_G17_Theme1\results/stats_analysis


### Data Related

In [ ]:
df = results_df.copy()

def get_n_issues(stats_str):
    if pd.isna(stats_str):
        return 0
    try:
        s = stats_str.replace('np.int64(', '').replace('np.float64(', '').replace(')', '')
        d = ast.literal_eval(s)
        return d.get('n_issues', 0)
    except:
        return 0

df['n_issues_cl'] = df['stats'].apply(get_n_issues)

df['rows_removed'] = df['n_before'] - df['n_after']
df['rows_affected'] = np.where(df['cleaning_name'] == '4_ModelAware',
                               df['n_issues_cl'],
                               df['rows_removed'])

df['pct_affected'] = (df['rows_affected'] / df['n_before']) * 100
df['retention_pct'] = (df['n_after'] / df['n_before']) * 100

pd.options.display.float_format = '{:.2f}'.format

# Table 6: Dataset Size Before vs After Cleaning
print("Generating Table 6: Dataset Size Before vs After Cleaning ...")
t_d1 = df.groupby('cleaning_name').agg({
    'n_before': 'mean',
    'n_after': 'mean',
    'rows_affected': 'mean'
}).reset_index()
t_d1.rename(columns={'rows_affected': 'rows_affected_or_removed'}, inplace=True)
file_name = os.path.join(STATS_DIR, 'table_06_size_impact.csv')
t_d1.to_csv(file_name, index=False)

# Table 7: Percentage of Data Affected
print("Generating Table 7: Percentage of Data Affected ...")
t_d2 = df.groupby('cleaning_name').agg({
    'pct_affected': 'mean'
}).reset_index()
file_name = os.path.join(STATS_DIR, 'table_07_pct_affected.csv')
t_d2.to_csv(file_name, index=False)

# Table 8: Corruption Severity vs Data Loss vs Accuracy
print("Generating Table 8: Corruption Severity vs Data Loss vs Accuracy ...")
corruption_impact = df[df['cleaning_name'] == '0_Corrupted'][['experiment', 'accuracy', 'pct_affected']].copy()
corruption_impact.rename(columns={'accuracy': 'corrupted_accuracy', 'pct_affected': 'intrinsic_data_loss_pct'}, inplace=True)

avg_cleaned_acc = df[df['cleaning_name'] != '0_Corrupted'].groupby('experiment')['accuracy'].mean().reset_index()
t_d3 = pd.merge(corruption_impact, avg_cleaned_acc, on='experiment')
t_d3.rename(columns={'accuracy': 'avg_cleaned_accuracy'}, inplace=True)
file_name = os.path.join(STATS_DIR, 'table_08_severity_vs_loss.csv')
t_d3.to_csv(file_name, index=False)

# Table 9: Data Retention vs Accuracy (Per Strategy)
print("Generating Table 9: Data Retention vs Accuracy (Per Strategy) ...")
t_d4 = df.groupby('cleaning_name').agg({
    'retention_pct': 'mean',
    'accuracy': 'mean'
}).reset_index()
t_d4.sort_values(by='accuracy', ascending=False, inplace=True)
file_name = os.path.join(STATS_DIR, 'table_09_retention_vs_accuracy.csv')
t_d4.to_csv(file_name, index=False)

print(f"\nData Analysis Tables generated and saved to: {STATS_DIR}")

del df, t_d1, t_d2, t_d3, t_d4
clear_memory()

Generating Table 6: Dataset Size Before vs After Cleaning ...
Generating Table 7: Percentage of Data Affected ...
Generating Table 8: Corruption Severity vs Data Loss vs Accuracy ...
Generating Table 9: Data Retention vs Accuracy (Per Strategy) ...

Data Analysis Tables generated and saved to: /Users/mansi/Documents/Acad/P3/data-preparation-project/results/stats_analysis
